# BoundaryCells cost calibration seed43
5 arms ×2 rounds, fixed20roundschedule; no utility claim. Every learned model DAGSNet; local records stay at their client. Cohort cell counts/targets only. W&B online before training; mandatory private credential workflow.

In [1]:

import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','wandb==0.29.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
assert wandb.login(key=_wandb_key,anonymous='never',relogin=True,verify=True)
os.environ['WANDB_API_KEY']=_wandb_key
del _wandb_key
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
opentelemetry-semantic-conventions 0.59b0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.45.1 which is incompatible.
wandb: WARNING The anonymous parameter to wandb.login() has no effect and will be removed in future versions.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] 

W&B authenticated online


In [2]:
MH=['__init__.py', 'archs.py', 'channel.py', 'dagsnet.py', 'metrics.py', 'mobility.py', 'model_packet.py', 'new_fl.py', 'new_methods.py', 'p2.py', 'pilot.py', 'prepare.py', 'teacher_base.py', 'teacher_driver.py', 'v10_energy.py', 'v10_run.py', 'v2_math.py', 'v2_methods.py', 'v2_run.py', 'v3_dense.py', 'v3_math.py', 'v3_methods.py', 'v3_run.py', 'v4_mala.py', 'v4_run.py', 'v4_schema.py', 'v4_support.py', 'v5_anchor.py', 'v5_books.py', 'v5_run.py', 'v6_geometry.py', 'v6_run.py', 'v6_schema.py', 'v7_basis.py', 'v7_nystrom.py', 'v7_run.py', 'v8_hybrid.py', 'v8_run.py', 'v9_anchor.py', 'v9_run.py']
PROJ=['boundary_driver.py', 'boundary_method.py', 'cells.py', 'probe.py']

import hashlib,json,shutil,time
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
assert torch.cuda.device_count()==2
assert all(torch.cuda.get_device_capability(i)==(7,5) for i in range(2))
assert torch.__version__=='2.10.0+cu128'
hits=[p.parent for p in Path('/kaggle/input').rglob('boundary_driver.py') if (p.parent/'train_0_X.npy').exists()]
assert len(hits)==1,hits
SRC=hits[0];WORK=Path('/kaggle/working/work');OUT=Path('/kaggle/working/runs')
SOURCE=Path('/kaggle/working/source');SOURCE.mkdir(exist_ok=True)
for folder in ('mh','knowledge','papers/pilot/fixture','papers/boundary-cells/proj'):
    (WORK/folder).mkdir(parents=True,exist_ok=True)
manifest=json.loads((SRC/'source_manifest.json').read_text())
for name,sha in manifest.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('source mismatch',name)
    shutil.copyfile(SRC/name,SOURCE/name)
for path in SRC.iterdir():
    if path.name in MH: dst=WORK/'mh'/path.name
    elif path.name in PROJ: dst=WORK/'papers/boundary-cells/proj'/path.name
    elif path.name=='meta.json': dst=WORK/'knowledge'/path.name
    elif path.name=='contract.md': continue
    else: dst=WORK/'papers/pilot/fixture'/path.name
    shutil.copyfile(path,dst)
fixture=json.loads((SRC/'manifest.json').read_text())
for name,sha in fixture['files'].items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('fixture mismatch',name)
shutil.copyfile(SRC/'manifest.json',SOURCE/'fixture_manifest.json')
shutil.copyfile(SRC/'source_manifest.json',SOURCE/'source_manifest.json')
print('source/fixture verified; CUDA',torch.__version__,flush=True)
sys.path.insert(0,str(WORK));sys.path.insert(0,str(WORK/'papers/boundary-cells/proj'))
from mh.dagsnet import build_model,CFG
from cells import packet,decode
import numpy as np
for gpu in (0,1):
    torch.cuda.set_device(gpu);torch.manual_seed(42)
    model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True).to(f'cuda:{gpu}').eval()
    x=torch.from_numpy(np.load(SRC/'train_0_X.npy',mmap_mode='r')[::100][:32].astype('float32')).to(f'cuda:{gpu}')
    z=model(x)[0];assert torch.isfinite(z).all()
    z.square().mean().backward();assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
    identity=bytes(16);encoded=packet(1,2,b'preflight',identity);assert decode(encoded,1,2,identity)==b'preflight'
    print('T4 forward/backward/codec PASS',gpu,flush=True)
    del model,x,z;torch.cuda.empty_cache()


source/fixture verified; CUDA 2.10.0+cu128
T4 forward/backward/codec PASS 0
T4 forward/backward/codec PASS 1


In [3]:
SEED=43

T0=time.time();OUT.mkdir(exist_ok=True)
methods=['bc_single','bc_direct2_fl','bc_direct4_fl','bc_teacher2_fl','bc_teacher4_fl']
pending=list(methods);active={};summary={};baseline_cost=None;seen={}
while active or pending:
    for gpu in (0,1):
        if gpu in active or not pending or time.time()-T0>1.5*3600:continue
        if baseline_cost is None and (pending[0]!='bc_single' or gpu!=0):continue
        arm=pending.pop(0);seed=SEED
        folder=OUT/f'{arm}_s{seed}';folder.mkdir(exist_ok=True)
        log=open(folder/'log.txt','w')
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),PYTHONPATH=str(WORK),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
        command=[sys.executable,str(WORK/'papers/boundary-cells/proj/boundary_driver.py'),
                 '--arm',arm,'--seed',str(seed),'--out',str(OUT),'--rounds','20',
                 '--stop_after','2','--eval_every','1','--aux_weight','.5','--temperature','1',
                 '--wandb_online','--max_hours','.5']
        active[gpu]=(arm,seed,subprocess.Popen(command,cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT),log)
        print('start calibration',arm,'seed',seed,'gpu',gpu,flush=True)
    for gpu,(arm,seed,proc,log) in list(active.items()):
        folder=OUT/f'{arm}_s{seed}';label=folder.name
        lines=(folder/'log.txt').read_text().splitlines()
        for line in lines[seen.get(label,0):]:
            if line.startswith(('setup','r','done','WANDB','Traceback','budget')) or 'Error' in line:
                print(label,'|',line,flush=True)
        seen[label]=len(lines)
        if proc.poll() is None:continue
        log.close();del active[gpu]
        file=folder/'result.json';result=json.loads(file.read_text()) if file.exists() else {}
        rows=result.get('history',[])
        if proc.returncode!=0 or len(rows)!=2:
            summary[label]={'rc':proc.returncode,'rounds':len(rows),'calibration_valid':False};continue
        seconds=[row['algorithm_seconds'] for row in rows]
        measured=sum(seconds)/2
        if arm=='bc_single':baseline_cost=measured
        ratio=measured/baseline_cost
        gate={'algorithm_seconds':seconds,'mean_seconds':measured,'baseline_mean_seconds':baseline_cost,
              'ratio':ratio,'pass_cost':ratio<=2,'physical_gpu':gpu,'seed':seed,'rounds':2,
              'round_schedule':20,'utility_measured':False}
        (folder/'cost_gate.json').write_text(json.dumps(gate,indent=2))
        summary[label]={'rc':0,'rounds':2,'calibration_valid':True,'cost':gate}
        print('COST',label,ratio,ratio<=2,flush=True)
    if not active and time.time()-T0>1.5*3600:break
    if not active and baseline_cost is None and pending and pending[0]!='bc_single':break
    if active or pending:time.sleep(2)
for arm in pending:summary[f'{arm}_s{SEED}']={'reason':'missing baseline or parent budget before start','calibration_valid':False}
(OUT/'calibration_summary.json').write_text(json.dumps(summary,indent=2))
print('CALIBRATION ONLY; no utility continuation',summary,flush=True)
assert len(summary)==5 and all(v.get('calibration_valid') for v in summary.values()),'incomplete calibration'


start calibration bc_single seed 43 gpu 0
bc_single_s43 | WANDB https://wandb.ai/21522798-uit/mobihet-boundary-cells/runs/bc_single_s43-a31f484f
bc_single_s43 | setup bc_single role private rows 1000369 sec 1.3
bc_single_s43 | r1 bc_single s43 private group gp 0.3804048985327022 loss 0.7992 66s
bc_single_s43 | r2 bc_single s43 private group gp 0.45205381418822316 loss 0.4337 63s
bc_single_s43 | done bc_single complete False
COST bc_single_s43 1.0 True
start calibration bc_direct2_fl seed 43 gpu 0
start calibration bc_direct4_fl seed 43 gpu 1
bc_direct2_fl_s43 | WANDB https://wandb.ai/21522798-uit/mobihet-boundary-cells/runs/bc_direct2_fl_s43-2fcd4462
bc_direct4_fl_s43 | WANDB https://wandb.ai/21522798-uit/mobihet-boundary-cells/runs/bc_direct4_fl_s43-1e33c743
bc_direct2_fl_s43 | setup bc_direct2_fl role private rows 1000369 sec 2.2
bc_direct4_fl_s43 | setup bc_direct4_fl role private rows 1000369 sec 2.3
bc_direct2_fl_s43 | r1 bc_direct2_fl s43 private group gp 0.3863218336738961 loss 